# Corrected FADC3D encoder — DS ON — seed 42 — 70/10/20 split

Fresh training run of the corrected FADC3D encoder with deep supervision
**enabled**, on a new patient-level 70/10/20 split generated with seed 42
and stratified by collection (DUKE / ISPY1 / ISPY2 / NACT).

**Contract**

- Split is patient-level, collection-stratified, deterministic under seed 42.
- The preprocessed 2-channel MAMA-MIA cache under
  `/kaggle/input/datasets/bharathkumarvemuri/mama-mia-preprocessed-cache-2ch`
  is treated as read-only: no `.npz` files are copied, renamed, or moved.
- The old `train/` and `val/` subfolder names refer to the previous
  experiment. This run reassigns every patient logically via a
  manifest CSV. Loaders read each `.npz` directly from its original path
  via that manifest.
- Fresh training only: no `--resume`, no reuse of any previous checkpoint,
  no reuse of the previous output directory. Preflight aborts if the new
  output directory already contains a checkpoint.
- Test set is **locked**. The RUN_FINAL_TEST cell is off by default and
  the training loader is never given access to the test partition.


In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
# Corrected FADC3D encoder, DS ON, fresh 70/10/20 split, seed 42.
# Do NOT modify any preserved hyperparameter without opening a new run.
SEED                = 42
GIT_BRANCH          = "feature/fadc3d-correct"
EXPECTED_GIT_COMMIT = "81cb3cdaeac6b8ab0a9cadf6d19e27503fedff4a"   # PIN — DS and nods runs share identical source


# NOTE: the Kaggle dataset owner slug is 'bharathkumarvemuri' for THIS run.
DATA_ROOT              = "/kaggle/input/datasets/bharathkumarvemuri/mama-mia-preprocessed-cache-2ch"
PREPROCESSED_CACHE_DIR = DATA_ROOT
CODE_DIR               = "/kaggle/working/FADC-3D"

# Separate output directory. Cannot collide with the previous DS
# (fadc3d_correct_encoder_s42) or nods (fadc3d_correct_encoder_nods_s42) runs.
OUTPUT_DIR_SMOKE  = "/kaggle/working/outputs/fadc3d_correct_encoder_ds_split701020_smoke"
OUTPUT_DIR_FULL   = "/kaggle/working/outputs/fadc3d_correct_encoder_ds_split701020_s42"

# Manifest paths live under the writable output dir on Kaggle.
MANIFEST_CSV      = f"{OUTPUT_DIR_FULL}/split_70_10_20_seed42.csv"
MANIFEST_META     = f"{OUTPUT_DIR_FULL}/split_70_10_20_seed42_metadata.json"

# Full-training hyperparameters (UNCHANGED vs. previous corrected-encoder runs).
EPOCHS         = 100
BATCH_SIZE     = 2
NUM_WORKERS    = 4
PATCH_SIZE     = [128, 128, 64]
LEARNING_RATE  = 1e-4
WARMUP_EPOCHS  = 5

# THE ABLATION KNOB FOR THIS EXPERIMENT.
DEEP_SUPERVISION = True

# Split ratios and seed (matched to manifest).
SPLIT_SEED       = 42
SPLIT_RATIOS     = (0.70, 0.10, 0.20)   # train, val, test

# k_att schedule (UNCHANGED).
K_ATT_TEMP_START    = 2.0
K_ATT_TEMP_END      = 1.0
K_ATT_ANNEAL_EPOCHS = 60

# Attention diversity aux DISABLED (UNCHANGED).
ATTN_DIVERSITY_WEIGHT = 0.0
# Position attention DISABLED (UNCHANGED default — --use_position_att not passed).
USE_POSITION_ATT      = False

# Smoke-test parameters.
SMOKE_PATCH_SIZE = [48, 48, 24]

MODEL_NAME = "unet3d_fadc_encoder_correct"

# ── VALIDATION SCHEDULE (formal only) ──────────────────────────────────
VAL_EVERY           = 20     # formal validation every N epochs
VAL_OVERLAP         = 0.5    # canonical evaluation overlap
VAL_SW_BATCH_SIZE   = 4
CHECKPOINT_EVERY    = 10

# ── FRESH TRAINING ONLY ────────────────────────────────────────────────
# No RESUME_FROM knob is defined; preflight aborts on any stale checkpoint
# under OUTPUT_DIR_FULL. If Kaggle times out mid-run, DO NOT relaunch
# without explicitly clearing OUTPUT_DIR_FULL first.

print(f"SEED                : {SEED}")
print(f"BRANCH              : {GIT_BRANCH}")
print(f"MODEL_NAME          : {MODEL_NAME}")
print(f"OUTPUT_DIR_FULL     : {OUTPUT_DIR_FULL}")
print(f"OUTPUT_DIR_SMOKE    : {OUTPUT_DIR_SMOKE}")
print(f"MANIFEST_CSV        : {MANIFEST_CSV}")
print(f"DEEP_SUPERVISION    : {DEEP_SUPERVISION}    <-- ON for this run")
print(f"SPLIT_SEED / RATIOS : {SPLIT_SEED} / {SPLIT_RATIOS}")
print(f"PATCH_SIZE (train)  : {PATCH_SIZE}")
print(f"EPOCHS / BATCH      : {EPOCHS} / {BATCH_SIZE}")
print()
print("VALIDATION SCHEDULE (formal only)")
print(f"  validation cases    : from manifest val partition (~10%)")
print(f"  formal overlap      : {VAL_OVERLAP}")
print(f"  validation frequency: every {VAL_EVERY} epochs")
print(f"  sw_batch_size       : {VAL_SW_BATCH_SIZE}")
print(f"  checkpoint_every    : {CHECKPOINT_EVERY}")
print()
print(f"k_att T             : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"attn_diversity_wt   : {ATTN_DIVERSITY_WEIGHT}  (disabled)")
print(f"use_position_att    : {USE_POSITION_ATT}   (disabled)")
print(f"RESUME              : disabled — fresh training only")


In [ ]:
# ── PREFLIGHT — ablation contract ──────────────────────────────────────
# Static assertions that must hold before ANY expensive cell runs. If any
# assertion fires, fix the config and re-run from the top — do not skip.
import os, re, sys

# 1) DEEP_SUPERVISION must be True.
assert DEEP_SUPERVISION is True, \
    f"PREFLIGHT: DEEP_SUPERVISION must be True for this run (got {DEEP_SUPERVISION!r})."

# 2) Output directory must be the new ds_split701020 variant.
assert "ds_split701020" in OUTPUT_DIR_FULL, \
    f"PREFLIGHT: OUTPUT_DIR_FULL must contain 'ds_split701020' (got {OUTPUT_DIR_FULL!r})."
for _forbidden in ("/outputs/fadc3d_correct_encoder_s42",
                   "/outputs/fadc3d_correct_encoder_nods_s42"):
    assert OUTPUT_DIR_FULL != _forbidden, \
        f"PREFLIGHT: OUTPUT_DIR_FULL collides with an earlier experiment: {_forbidden}"

# 3) No RESUME_FROM must be defined.
assert "RESUME_FROM" not in globals() or not globals().get("RESUME_FROM"), \
    "PREFLIGHT: RESUME_FROM must be unset/empty; this run is fresh-only."

# 4) Model / split identity sanity.
assert MODEL_NAME == "unet3d_fadc_encoder_correct", MODEL_NAME
assert abs(sum(SPLIT_RATIOS) - 1.0) < 1e-9, SPLIT_RATIOS
assert abs(SPLIT_RATIOS[0] - 0.70) < 1e-9 and abs(SPLIT_RATIOS[1] - 0.10) < 1e-9 \
       and abs(SPLIT_RATIOS[2] - 0.20) < 1e-9, \
    f"PREFLIGHT: SPLIT_RATIOS must be exactly (0.70, 0.10, 0.20), got {SPLIT_RATIOS}"

# 5) Training-command preview: the argv we'll feed to the training script
#    below must include --deep_supervision, --split_manifest and MUST NOT
#    include --resume.
_training_cmd_preview = [
    sys.executable, "-u", "training/train_centralized_correct.py",
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_FULL,
    "--epochs",             str(EPOCHS),
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        str(NUM_WORKERS),
    "--patch_size",         str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      str(WARMUP_EPOCHS),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",str(K_ATT_ANNEAL_EPOCHS),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--split_manifest",     MANIFEST_CSV,
    "--val_every",          str(VAL_EVERY),
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--checkpoint_every",   str(CHECKPOINT_EVERY),
    "--deep_supervision",
]
assert "--deep_supervision" in _training_cmd_preview, \
    "PREFLIGHT: training command must contain --deep_supervision for this run."
assert "--resume" not in _training_cmd_preview, \
    "PREFLIGHT: training command must NOT contain --resume."
assert "--split_manifest" in _training_cmd_preview, \
    "PREFLIGHT: training command must contain --split_manifest."

# 6) VAL_OVERLAP contract — formal protocol is 0.5.
assert abs(VAL_OVERLAP - 0.5) < 1e-9, f"PREFLIGHT: VAL_OVERLAP must be 0.5 (got {VAL_OVERLAP!r})."

# 7) Read-only source cache must exist (Kaggle mount).
if not os.path.isdir(PREPROCESSED_CACHE_DIR):
    print(f"NOTE: PREPROCESSED_CACHE_DIR not mounted locally ({PREPROCESSED_CACHE_DIR}). "
          "Kaggle will exercise the check.")

# 8) Adaptive-conv contract — enforced later by the arch cell.
_expected_adaptive_convs = 8

# 9) Guard against auto-resume from a stale checkpoint that might already
#    live under OUTPUT_DIR_FULL. Refuse to continue if any .pth is present.
_stale = []
if os.path.isdir(OUTPUT_DIR_FULL):
    for name in os.listdir(OUTPUT_DIR_FULL):
        if name.endswith(".pth"):
            _stale.append(os.path.join(OUTPUT_DIR_FULL, name))
if _stale:
    raise SystemExit(
        "PREFLIGHT: OUTPUT_DIR_FULL already contains checkpoint files:\n  "
        + "\n  ".join(_stale)
        + "\n\nThis run is fresh-only. Delete these files (or point "
          "OUTPUT_DIR_FULL at a clean directory) and re-run the CONFIG cell."
    )

# 10) Guard against re-using the previous DS output dir by mistake.
for _forbidden_dir in ("/kaggle/working/outputs/fadc3d_correct_encoder_s42",
                       "/kaggle/working/outputs/fadc3d_correct_encoder_nods_s42"):
    assert OUTPUT_DIR_FULL != _forbidden_dir, \
        f"PREFLIGHT: OUTPUT_DIR_FULL collides with prior run {_forbidden_dir}"

print("PREFLIGHT OK")
print(f"  DEEP_SUPERVISION       : {DEEP_SUPERVISION}")
print(f"  OUTPUT_DIR_FULL        : {OUTPUT_DIR_FULL}")
print(f"  MODEL_NAME             : {MODEL_NAME}")
print(f"  adaptive_convs contract: {_expected_adaptive_convs}")
print(f"  SPLIT_RATIOS           : {SPLIT_RATIOS}  seed={SPLIT_SEED}")
print(f"  VAL_OVERLAP            : {VAL_OVERLAP}")
print(f"  --deep_supervision     : IN training command")
print(f"  --split_manifest       : IN training command")
print(f"  --resume               : NOT in training command")
print(f"  stale checkpoints      : none under OUTPUT_DIR_FULL")


In [ ]:
# ── 1. INSTALL DEPS + REQUIRE CUDA ─────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA is not available on this session. Switch the accelerator "
        "to GPU (P100 or T4x2 — code uses only cuda:0)."
    )
print(f"GPU (cuda:0)   : {torch.cuda.get_device_name(0)}")
p = torch.cuda.get_device_properties(0)
print(f"VRAM (cuda:0)  : {p.total_memory / 1e9:.1f} GB")


In [ ]:
# ── 2. CLONE / CHECKOUT feature/fadc3d-correct AT PINNED COMMIT ────────
# Pins the source code to EXPECTED_GIT_COMMIT so DS and no-DS ablations
# run byte-identical training code. Detached-HEAD checkout on purpose —
# we never `git pull` after pinning, because a downstream commit could
# change the training loop without us noticing.
import os, sys, subprocess

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        sys.stdout.write(r.stdout); sys.stderr.write(r.stderr)
        raise SystemExit(f"Command failed ({r.returncode}): {' '.join(cmd)}")
    return r.stdout.strip()

if os.path.exists(CODE_DIR):
    print(f"repo present; fetching {GIT_BRANCH} ...")
    _run(["git", "-C", CODE_DIR, "fetch", "--all", "--tags"])
else:
    _run(["git", "clone", "-b", GIT_BRANCH,
          "https://github.com/Vemuri-BK/FADC-3D.git", CODE_DIR])
    _run(["git", "-C", CODE_DIR, "fetch", "--all", "--tags"])

# Detached-HEAD checkout at the exact pinned commit. Never pull.
_run(["git", "-C", CODE_DIR, "checkout", "--detach", EXPECTED_GIT_COMMIT])
GIT_COMMIT_HASH = _run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"])
GIT_COMMIT_LINE = _run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"])

assert GIT_COMMIT_HASH == EXPECTED_GIT_COMMIT, (
    f"PINNED CHECKOUT FAILED. expected={EXPECTED_GIT_COMMIT!r} "
    f"got={GIT_COMMIT_HASH!r}. Refusing to continue — the DS and no-DS "
    "runs must share identical source code."
)
print(f"pinned commit  : {EXPECTED_GIT_COMMIT}")
print(f"HEAD           : {GIT_COMMIT_LINE}")
print(f"HEAD (verified): {GIT_COMMIT_HASH}")

sys.path.insert(0, CODE_DIR)

os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)
with open(os.path.join(OUTPUT_DIR_FULL, "source_commit.txt"), "w", encoding="utf-8") as f:
    f.write(GIT_COMMIT_HASH + "\n" + GIT_COMMIT_LINE + "\n")

for p in ("fadc_3d_correct/adaptive_dilated_conv_3d.py",
          "fadc_3d_correct/ada_kernel_3d.py",
          "fadc_3d_correct/freq_select_3d.py",
          "models/unet_3d_fadc_correct.py",
          "training/train_centralized_correct.py",
          "training/split_manifest.py",
          "training/evaluate_correct_checkpoint.py",
          "tests/test_fadc_3d_correct.py",
          "tests/test_train_correct_utils.py",
          "tests/test_split_manifest.py",
          "diag_fadc_3d_correct.py"):
    assert os.path.exists(os.path.join(CODE_DIR, p)), f"missing on branch: {p}"
print("Corrected FADC3D + split-manifest + diag files present on branch.")


In [ ]:
# ── 3. SPLIT GENERATION — deterministic seed-42 collection-stratified 70/10/20 ─
# Reads the read-only .npz cache in place; writes only the manifest CSV +
# metadata JSON under OUTPUT_DIR_FULL. Never copies / moves the .npz files.
import os, sys, json, csv
sys.path.insert(0, CODE_DIR)

from training.split_manifest import (
    generate_and_write, load_manifest, verify_manifest_partitions,
    manifest_sha256, COLLECTIONS,
)

os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)

# If a manifest already exists, refuse to overwrite it silently — a locked
# manifest is a contract; regenerating it would invalidate every downstream
# checkpoint's split_identity.
if os.path.exists(MANIFEST_CSV):
    print(f"MANIFEST_CSV already exists: {MANIFEST_CSV}")
    print("  -> not regenerating; existing manifest is authoritative.")
    with open(MANIFEST_META, encoding="utf-8") as f:
        meta = json.load(f)
else:
    meta = generate_and_write(
        cache_root=PREPROCESSED_CACHE_DIR,
        csv_path=MANIFEST_CSV, meta_path=MANIFEST_META,
        seed=SPLIT_SEED, ratios=SPLIT_RATIOS,
    )

# Integrity + partition preflight.
partition_summary = verify_manifest_partitions(MANIFEST_CSV)
MANIFEST_SHA256 = manifest_sha256(MANIFEST_CSV)
assert MANIFEST_SHA256 == meta["csv_sha256"], "metadata SHA256 does not match CSV on disk!"

train_cases = load_manifest(MANIFEST_CSV, split="train", cache_root=PREPROCESSED_CACHE_DIR)
val_cases   = load_manifest(MANIFEST_CSV, split="val",   cache_root=PREPROCESSED_CACHE_DIR)
test_cases  = load_manifest(MANIFEST_CSV, split="test",  cache_root=PREPROCESSED_CACHE_DIR)

n_total = partition_summary["n_total"]
n_train, n_val, n_test = partition_summary["n_train"], partition_summary["n_val"], partition_summary["n_test"]
r_train_actual = n_train / n_total
r_val_actual   = n_val   / n_total
r_test_actual  = n_test  / n_total

# Ratios approximately correct (±1% absolute is generous even with stratification).
assert abs(r_train_actual - SPLIT_RATIOS[0]) < 0.02
assert abs(r_val_actual   - SPLIT_RATIOS[1]) < 0.02
assert abs(r_test_actual  - SPLIT_RATIOS[2]) < 0.02

# Pairwise disjointness enforced by the manifest generator; re-assert here.
train_ids = {c["patient_id"] for c in train_cases}
val_ids   = {c["patient_id"] for c in val_cases}
test_ids  = {c["patient_id"] for c in test_cases}
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert len(train_ids | val_ids | test_ids) == n_total

# All four collections represented in every split (where possible).
for coll, per_split in meta["n_per_collection_split"].items():
    n_coll = sum(per_split.values())
    if n_coll > 0:
        assert per_split["train"] > 0, f"train missing collection {coll}"
        assert per_split["val"]   > 0, f"val missing collection {coll}"
        assert per_split["test"]  > 0, f"test missing collection {coll}"

# Verify each referenced .npz exists (load_manifest with require_exists=True
# would have raised, but re-assert for defence-in-depth).
for c in train_cases + val_cases + test_cases:
    assert os.path.exists(c["npz_path"]), c["npz_path"]

print("SPLIT GENERATION OK")
print(f"  seed                : {SPLIT_SEED}   ratios: {SPLIT_RATIOS}")
print(f"  total patients      : {n_total}")
print(f"  train / val / test  : {n_train} / {n_val} / {n_test}   "
      f"actual ratio: {r_train_actual:.3f} / {r_val_actual:.3f} / {r_test_actual:.3f}")
print(f"  manifest CSV        : {MANIFEST_CSV}")
print(f"  manifest metadata   : {MANIFEST_META}")
print()
print("=" * 70)
print("!! CANONICAL DS MANIFEST SHA256 — PASTE THIS INTO THE NODS NOTEBOOK'S")
print("!! EXPECTED_MANIFEST_SHA256 CONFIG FIELD BEFORE LAUNCHING THE NODS RUN.")
print(f"    {MANIFEST_SHA256}")
print("=" * 70)
print()
print(f"  manifest SHA256     : {MANIFEST_SHA256}")
print()
print("Per-collection split counts:")
for coll in COLLECTIONS:
    ps = meta["n_per_collection_split"].get(coll, {})
    n_c = sum(ps.values())
    if n_c:
        print(f"  {coll:6s}  n={n_c:4d}  train={ps.get('train',0):4d}  "
              f"val={ps.get('val',0):3d}  test={ps.get('test',0):4d}")
print()
print("5 example patient_ids per split:")
for name, ids in (("train", sorted(train_ids)),
                  ("val",   sorted(val_ids)),
                  ("test",  sorted(test_ids))):
    print(f"  {name:5s}  {ids[:5]}")


In [ ]:
# ── 4. PULL VERIFIER + ARCHITECTURE CHECKS ─────────────────────────────
import gc, sys, torch
sys.path.insert(0, CODE_DIR)

from fadc_3d_correct.adaptive_dilated_conv_3d import AdaptiveDilatedConv3D
from models.unet_3d_fadc_correct import (
    build_unet3d_fadc_correct, EXPECTED_ADAPTIVE_CONV_COUNT, MODEL_NAMES,
)

assert AdaptiveDilatedConv3D.KERNEL_SIZE == 3
assert MODEL_NAME in MODEL_NAMES, f"MODEL_NAME {MODEL_NAME} not in {MODEL_NAMES}"

m = build_unet3d_fadc_correct(MODEL_NAME, in_channels=2, out_channels=2,
                              base_filters=32, deep_supervision=DEEP_SUPERVISION).cuda()

n_adapt = m.count_adaptive_convs()
assert n_adapt == EXPECTED_ADAPTIVE_CONV_COUNT["encoder"] == 8, \
    f"encoder placement must yield 8 adaptive convs, got {n_adapt}"
adapt_names = m.adaptive_conv_names()
outside_enc = [n for n in adapt_names if not n.startswith("enc")]
assert not outside_enc, f"adaptive convs found outside enc*: {outside_enc}"

sample = next(mm for mm in m.modules() if isinstance(mm, AdaptiveDilatedConv3D))
assert sample.dilation_list == (1, 2, 3), sample.dilation_list

# DS training-mode output must be a tuple of >=2 tensors.
m.train()
x = torch.randn(1, 2, 32, 32, 16, device="cuda")
y_train = m(x)
assert isinstance(y_train, tuple) and len(y_train) >= 2, \
    f"expected DS tuple in train() mode, got {type(y_train).__name__} len={len(y_train) if isinstance(y_train,(tuple,list)) else 'n/a'}"
# Eval-mode primary head should still return a single tensor.
m.eval()
with torch.no_grad():
    y_eval = m(x)
    # Note: our model returns a tuple always if DS enabled; the *predictor*
    # wrapper picks head 0 during sliding-window inference. The wrapper is
    # exercised in the smoke reload cell below.

print(f"MODEL_NAME       : {MODEL_NAME}")
print(f"adaptive convs   : {n_adapt}/8 (all under enc*)")
print(f"params           : {sum(p.numel() for p in m.parameters()):,}")
print(f"DS train output  : tuple(len={len(y_train)}), main shape {tuple(y_train[0].shape)}")
print("Pull + architecture checks passed.")

del sample, m, x, y_train, y_eval
gc.collect(); torch.cuda.empty_cache()
print(f"post-check VRAM alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── 5. RUN CORRECTNESS TESTS (abort on nonzero) ───────────────────────
import os, subprocess, sys

TEST_FILES = [
    "tests/test_fadc_3d_correct.py",       # FADC3D math correctness
    "tests/test_train_correct_utils.py",   # training loop utilities
    "tests/test_split_manifest.py",        # NEW — split-manifest logic
]

for rel in TEST_FILES:
    test_path = os.path.join(CODE_DIR, rel)
    assert os.path.exists(test_path), f"test file missing: {test_path}"
    print(f"---- running {rel} ----")
    res = subprocess.run([sys.executable, test_path], cwd=CODE_DIR,
                         capture_output=True, text=True)
    print(res.stdout[-6000:])
    if res.returncode != 0:
        sys.stderr.write(res.stderr[-3000:])
        raise SystemExit(f"Test file {rel} FAILED (exit {res.returncode}) — refusing to launch training.")
    print(f"---- {rel} PASSED ----\n")
print("All correctness tests PASSED.")


In [ ]:
# ── 6. SMOKE TRAINING — a handful of training + val cases, DS on ──────
# Uses the manifest so smoke exercises the same partitioning code path as
# full training. The trainer's --smoke_test flag caps to 4 training cases
# and forces val_every=1. TEST cases are NEVER accessed here — the
# training script never asks for the test partition.
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_SMOKE, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_SMOKE,
    "--patch_size",         str(SMOKE_PATCH_SIZE[0]), str(SMOKE_PATCH_SIZE[1]), str(SMOKE_PATCH_SIZE[2]),
    "--batch_size",         "2",
    "--warmup_epochs",      "1",
    "--val_every",          "1",
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs","1",
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--split_manifest",     MANIFEST_CSV,
    "--smoke_test",
    "--deep_supervision",
]
assert DEEP_SUPERVISION is True, "This notebook trains with deep supervision ON."
assert "--deep_supervision" in cmd,   "SMOKE: --deep_supervision must be in cmd."
assert "--resume" not in cmd,         "SMOKE: --resume must NOT be in cmd."
assert "--split_manifest" in cmd,     "SMOKE: --split_manifest must be in cmd."

print("SMOKE command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nSMOKE exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"SMOKE training failed (exit {proc.returncode}). Refusing to launch full training.")


In [ ]:
# ── 7. SMOKE CKPT: strict=True reload + DS + manifest identity checks ─
# Verifies that the smoke checkpoint (a) round-trips under strict=True,
# (b) declares deep_supervision=True in arch_identity, (c) returns a tuple
# in train() mode (DS heads live), (d) reports a single-tensor primary
# output through make_primary_predictor (what sliding-window inference
# consumes), (e) carries the split_manifest_sha256 we generated above.
import os, sys, torch
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct
from training.train_centralized_correct import make_primary_predictor

ckpt_path = os.path.join(OUTPUT_DIR_SMOKE, "last_checkpoint.pth")
assert os.path.exists(ckpt_path), f"smoke checkpoint missing: {ckpt_path}"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"epoch      : {ckpt.get('epoch')}")
print(f"best_dice  : {ckpt.get('best_dice')}")
arch = ckpt.get("arch_identity")
print(f"arch_id    : {arch}")
split_id = ckpt.get("split_identity")
print(f"split_id   : {split_id}")

assert arch is not None, "smoke checkpoint missing arch_identity"
assert arch["deep_supervision"] is True, \
    f"smoke ckpt arch_identity['deep_supervision'] must be True, got {arch['deep_supervision']!r}"
assert arch["model_name"] == "unet3d_fadc_encoder_correct"
assert split_id and split_id.get("split_manifest_sha256") == MANIFEST_SHA256, \
    f"smoke ckpt split_manifest_sha256 mismatch: {split_id!r} vs {MANIFEST_SHA256!r}"

model = build_unet3d_fadc_correct(
    arch["model_name"],
    in_channels=arch["in_channels"], out_channels=arch["out_channels"],
    base_filters=arch["base_filters"], deep_supervision=arch["deep_supervision"],
)
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
print(f"strict load: missing={len(missing)} unexpected={len(unexpected)}")
assert not missing and not unexpected, "strict reload mismatch"

n_adapt = model.count_adaptive_convs()
assert n_adapt == 8, f"reloaded model has {n_adapt} adaptive convs, expected 8"

# DS train-mode output must be a tuple.
model.train()
x = torch.randn(1, 2, 32, 32, 16)
y_train = model(x)
assert isinstance(y_train, tuple) and len(y_train) >= 2, \
    f"expected DS tuple in train() mode, got {type(y_train).__name__}"
assert torch.isfinite(y_train[0]).all(), "train() main head produced NaN/Inf"

# make_primary_predictor should return a single tensor (what SW uses).
model.eval()
predictor = make_primary_predictor(model)
with torch.no_grad():
    y_pred = predictor(x)
assert not isinstance(y_pred, (tuple, list)), \
    f"predictor returned {type(y_pred).__name__} — expected a single tensor"
print(f"train() output    : tuple(len={len(y_train)}) main shape {tuple(y_train[0].shape)}")
print(f"eval predictor    : tensor shape {tuple(y_pred.shape)}")

# Backward + optimizer step — verify the DS loss surface is finite.
opt = torch.optim.SGD(model.parameters(), lr=1e-5)
opt.zero_grad()
loss = sum(t.float().pow(2).mean() for t in y_train)
loss.backward()
opt.step()
assert torch.isfinite(loss).item(), "DS backward produced non-finite loss"
print(f"backward + opt.step OK  DS loss={loss.item():.4e}")


In [ ]:
# ── 8. GPU MEMORY PROBE — one 128x128x64 batch WITH every DS head ─────
import torch, gc, sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct
from torch.amp import autocast, GradScaler


def run_gpu_memory_probe() -> float:
    torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
    print(f"pre-probe VRAM alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    model = build_unet3d_fadc_correct(
        MODEL_NAME, in_channels=2, out_channels=2, base_filters=32,
        deep_supervision=DEEP_SUPERVISION,
    ).cuda().train()
    scaler = GradScaler("cuda")
    x = torch.randn(BATCH_SIZE, 2, *PATCH_SIZE, device="cuda")
    with autocast("cuda"):
        y = model(x)
        outputs = y if isinstance(y, tuple) else (y,)
        loss = sum(o.float().pow(2).mean() for o in outputs)
    scaler.scale(loss).backward()
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"probe OK. n_outputs_in_loss={len(outputs)} peak VRAM alloc: {peak:.2f} GB")
    return peak

try:
    _peak = run_gpu_memory_probe()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        _peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"OOM at probe. peak alloc: {_peak:.2f} GB")
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        raise SystemExit("Full-size probe OOMed. Not launching full training.")
    raise

gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f"post-cleanup memory_allocated : {torch.cuda.memory_allocated()/1e9:.3f} GB")
print(f"post-cleanup memory_reserved  : {torch.cuda.memory_reserved()/1e9:.3f} GB")


In [ ]:
# ── 9. FULL TRAINING — 100 ep, DS on, manifest-driven, fresh-only ─────
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_FULL,
    "--epochs",             str(EPOCHS),
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        str(NUM_WORKERS),
    "--patch_size",         str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      str(WARMUP_EPOCHS),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",str(K_ATT_ANNEAL_EPOCHS),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--split_manifest",     MANIFEST_CSV,
    "--val_every",          str(VAL_EVERY),
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--checkpoint_every",   str(CHECKPOINT_EVERY),
    "--deep_supervision",
]

# Contract checks — hard fail before any launch if a required flag is
# missing or a forbidden one snuck back in via editor drift.
assert DEEP_SUPERVISION is True, "DS notebook must run with DS on."
assert "--deep_supervision" in cmd, "Refusing to launch: --deep_supervision missing."
assert "--split_manifest"   in cmd, "Refusing to launch: --split_manifest missing."
assert "--resume"       not in cmd, "Refusing to launch: --resume leaked into cmd."

# Fresh-only guard: re-check for stray .pth files in OUTPUT_DIR_FULL just
# before subprocess launch (preflight ran earlier; defence-in-depth).
_existing = [n for n in (os.listdir(OUTPUT_DIR_FULL) if os.path.isdir(OUTPUT_DIR_FULL) else [])
             if n.endswith(".pth")]
if _existing:
    raise SystemExit(
        f"Refusing to launch: OUTPUT_DIR_FULL already contains {_existing}. "
        "This run is fresh-only."
    )

print("FRESH mode: starting from epoch 1 (no --resume passed)")
print(f"Manifest CSV       : {MANIFEST_CSV}")
print(f"Manifest SHA256    : {MANIFEST_SHA256}")
print("FULL command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nFULL exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"FULL training exited nonzero ({proc.returncode}).")


In [ ]:
# ── 10. STANDALONE FORMAL VALIDATION on best_model.pth ────────────────
# Read-only re-computation of validation Dice/IoU/Sens on the val split.
# Same protocol as the training-loop validator; a sanity check that the
# best_model.pth we selected on validation Dice is reproducible.
import os, subprocess, sys

best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
if not os.path.exists(best_ckpt):
    print(f"(no best_model.pth at {best_ckpt}; skipping)")
else:
    out_json = os.path.join(OUTPUT_DIR_FULL, "val_eval_bestmodel.json")
    cmd = [
        sys.executable, "-u",
        os.path.join(CODE_DIR, "training", "evaluate_correct_checkpoint.py"),
        "--checkpoint",             best_ckpt,
        "--data_root",              DATA_ROOT,
        "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
        "--patch_size",             str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
        "--overlap",                str(VAL_OVERLAP),
        "--sw_batch_size",          str(VAL_SW_BATCH_SIZE),
        "--num_workers",            str(NUM_WORKERS),
        "--split_manifest",         MANIFEST_CSV,
        "--split_partition",        "val",
        "--require_manifest_checksum", MANIFEST_SHA256,
        "--per_collection",
        "--out",                    out_json,
    ]
    print("VAL eval command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    while True:
        chunk = proc.stdout.read(512)
        if not chunk: break
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(
            f"Formal-val subprocess failed (exit {proc.returncode}). "
            "Refusing to trust the printed metrics."
        )
    print(f"\nVAL eval exit code: {proc.returncode}")
    print(f"metrics written to: {out_json}")


In [ ]:
# ── 11. REAL-MRI DIAGNOSTIC ON best_model.pth — MANIFEST-RESTRICTED ────
# The diagnostic loads real MRI patches from ONLY the val partition of
# the manifest. It never scans the physical directory and cannot touch
# test patients even though test cases share the same physical folders.
#
# Pre-call disjointness assertion + subprocess-returncode check make this
# the sole load path — a nonzero exit or a stray test/train patient would
# raise SystemExit long before any results are trusted.
import os, sys, subprocess

best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")

# Manifest-partition IDs — used both to sanity-check disjointness and to
# confirm the diagnostic sees only val patients.
_train_ids = {c["patient_id"] for c in load_manifest(MANIFEST_CSV,
                                                     split="train",
                                                     cache_root=PREPROCESSED_CACHE_DIR)}
_val_ids   = {c["patient_id"] for c in load_manifest(MANIFEST_CSV,
                                                     split="val",
                                                     cache_root=PREPROCESSED_CACHE_DIR)}
_test_ids  = {c["patient_id"] for c in load_manifest(MANIFEST_CSV,
                                                     split="test",
                                                     cache_root=PREPROCESSED_CACHE_DIR)}
assert _val_ids.isdisjoint(_train_ids), "val ∩ train non-empty"
assert _val_ids.isdisjoint(_test_ids),  "val ∩ test  non-empty"
assert _train_ids.isdisjoint(_test_ids), "train ∩ test non-empty"
print(f"partition set sizes: train={len(_train_ids)}  val={len(_val_ids)}  test={len(_test_ids)}")

if not os.path.exists(best_ckpt):
    print(f"(no best_model.pth at {best_ckpt}; skipping diag)")
elif not _val_ids:
    print("(val partition empty; skipping diag)")
else:
    cmd = [
        sys.executable, os.path.join(CODE_DIR, "diag_fadc_3d_correct.py"),
        "--ckpt", best_ckpt,
        "--split_manifest",         MANIFEST_CSV,
        "--split_partition",        "val",
        "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
        "--n_patches", "4",
        "--patch_size", "96", "96", "48",
    ]
    print("DIAG command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, bufsize=0)
    captured_lines = []
    while True:
        chunk = proc.stdout.read(512)
        if not chunk: break
        text = chunk.decode("utf-8", errors="replace")
        captured_lines.append(text)
        sys.stdout.write(text); sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(
            f"Diagnostic subprocess failed (exit {proc.returncode}). "
            "Refusing to trust the printed statistics."
        )

    # Post-call parse: extract every printed patient_id and assert it
    # belongs to val (defence-in-depth against any future refactor of the
    # diagnostic script that might silently change its selection scope).
    _joined = "".join(captured_lines)
    import re as _re
    _printed_ids = set(_re.findall(r"patient_id=([^\s]+)", _joined))
    _test_hits  = _printed_ids & _test_ids
    _train_hits = _printed_ids & _train_ids
    _val_hits   = _printed_ids & _val_ids
    assert not _test_hits,  f"LEAK: diag touched test patients: {sorted(_test_hits)}"
    assert not _train_hits, f"LEAK: diag touched train patients: {sorted(_train_hits)}"
    assert _printed_ids and _printed_ids == _val_hits, (
        f"diag selection is not a strict subset of val: printed={sorted(_printed_ids)} "
        f"val_hits={sorted(_val_hits)}"
    )
    print()
    print(f"DIAG selection OK — {len(_printed_ids)} patient(s), all in val partition.")
    print(f"  patient_ids : {sorted(_printed_ids)}")


In [ ]:
# ── 12. FINAL INTERNAL TEST — LOCKED OFF BY DEFAULT ────────────────────
# The final test partition is evaluated exactly ONCE per model, and only
# AFTER training is complete. Set RUN_FINAL_TEST = True manually in this
# cell (not upstream in CONFIG) and re-run this cell only.
#
# Contract:
#   - Loads best_model.pth (selected during training by validation Dice).
#   - Loads ONLY the manifest's 'test' partition (20%).
#   - model.eval() + torch.no_grad(), formal sliding-window inference,
#     patch_size=128x128x64, overlap=0.5, sw_batch_size=4.
#   - NO threshold tuning, NO TTA, NO post-processing using test labels.
#   - NEVER updates the checkpoint. Writes JSON + per-patient CSV +
#     per-collection CSV under OUTPUT_DIR_FULL.

RUN_FINAL_TEST = False   # <-- change to True after training completes.

import os, sys, json, csv, subprocess

if not RUN_FINAL_TEST:
    print("RUN_FINAL_TEST is False. Skipping final-test evaluation.")
    print("To run the final test:")
    print("  1) confirm training is complete and best_model.pth exists,")
    print("  2) set RUN_FINAL_TEST = True in this cell (do NOT edit CONFIG),")
    print("  3) re-run this cell only.")
else:
    best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
    if not os.path.exists(best_ckpt):
        raise SystemExit(
            f"Refusing to run final test: {best_ckpt} does not exist. "
            "Training must finish first."
        )

    out_json     = os.path.join(OUTPUT_DIR_FULL, "final_test_metrics.json")
    out_per_pat  = os.path.join(OUTPUT_DIR_FULL, "final_test_per_patient.csv")
    out_per_coll = os.path.join(OUTPUT_DIR_FULL, "final_test_per_collection.csv")

    cmd = [
        sys.executable, "-u",
        os.path.join(CODE_DIR, "training", "evaluate_correct_checkpoint.py"),
        "--checkpoint",             best_ckpt,
        "--data_root",              DATA_ROOT,
        "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
        "--patch_size",             str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
        "--overlap",                str(VAL_OVERLAP),
        "--sw_batch_size",          str(VAL_SW_BATCH_SIZE),
        "--num_workers",            str(NUM_WORKERS),
        "--split_manifest",         MANIFEST_CSV,
        "--split_partition",        "test",
        "--require_manifest_checksum", MANIFEST_SHA256,
        "--per_collection",
        "--out",                    out_json,
    ]
    print("FINAL INTERNAL TEST command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    while True:
        chunk = proc.stdout.read(512)
        if not chunk: break
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"Final test evaluator exited nonzero ({proc.returncode}).")

    # Post-process the JSON into two CSVs for easy inspection.
    with open(out_json, encoding="utf-8") as f:
        result = json.load(f)

    with open(out_per_pat, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(["patient_id", "collection", "dice", "iou", "sensitivity"])
        for r in result.get("per_case_metrics", []):
            w.writerow([r.get("patient_id"), r.get("collection"),
                        r.get("dice"), r.get("iou"), r.get("sensitivity")])
    with open(out_per_coll, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(["collection", "n", "dice_mean", "dice_median",
                    "dice_std", "iou_mean", "sens_mean"])
        for coll, d in (result.get("per_collection") or {}).items():
            w.writerow([coll, d["n"], d["dice_mean"], d["dice_median"],
                        d["dice_std"], d["iou_mean"], d["sens_mean"]])

    print()
    print("FINAL INTERNAL TEST RESULT")
    print(f"  n_test patients          : {result['n_cases']}")
    print(f"  mean patient-level Dice  : {result['dice']:.4f}")
    print(f"  median patient-level Dice: {result['dice_median']:.4f}")
    print(f"  std                      : {result['dice_std']:.4f}")
    print(f"  95% CI                   : [{result['dice_ci95'][0]:.4f}, {result['dice_ci95'][1]:.4f}]")
    print(f"  IoU                      : {result['iou']:.4f}")
    print(f"  Sensitivity              : {result['sensitivity']:.4f}")
    if result.get("per_collection"):
        print("  Per-collection (dice_mean):")
        for coll, d in result["per_collection"].items():
            print(f"    {coll:8s} n={d['n']:4d}  dice_mean={d['dice_mean']:.4f}  "
                  f"iou_mean={d['iou_mean']:.4f}  sens_mean={d['sens_mean']:.4f}")
    print(f"  checkpoint               : {result['checkpoint']}")
    print(f"  ckpt arch_identity       : {result['arch_identity']}")
    print(f"  ckpt split_identity      : {result['split_identity']}")
    print(f"  manifest SHA256 verified : {MANIFEST_SHA256}")
    print(f"  per-patient CSV          : {out_per_pat}")
    print(f"  per-collection CSV       : {out_per_coll}")


In [ ]:
# ── 13. DOWNLOAD LINKS ────────────────────────────────────────────────
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "last_checkpoint.pth",
              "train_log.json", "meta.json", "source_commit.txt",
              "split_70_10_20_seed42.csv",
              "split_70_10_20_seed42_metadata.json",
              "val_eval_bestmodel.json",
              "final_test_metrics.json",
              "final_test_per_patient.csv",
              "final_test_per_collection.csv"):
    p = os.path.join(OUTPUT_DIR_FULL, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")
